# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

In [ ]:
import pandas as pd

# Cargar los datos
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Limpiar nombres de columnas: minúsculas y "_" en vez de espacios
df.columns = df.columns.str.lower().str.replace(" ", "_")

# Quitar la columna índice sobrante si existe
df = df.drop(columns=[c for c in df.columns if c.startswith("unnamed")])

# Crear la columna "month" a partir de la fecha si no existe
if "month" not in df.columns:
    df["month"] = pd.to_datetime(df["effective_to_date"], format="%m/%d/%y").dt.month

print(df.shape)
df.head()

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [ ]:
# Filtro con dos condiciones unidas con & (cada una entre paréntesis)
low_claim_yes = df[(df["total_claim_amount"] < 1000) & (df["response"] == "Yes")]

print("Clientes que cumplen las dos condiciones:", low_claim_yes.shape[0])
low_claim_yes.head()

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [ ]:
# Solo clientes que respondieron "Yes"
yes = df[df["response"] == "Yes"]

# Media de prima mensual, CLV y reclamaciones por tipo de póliza y género
segmentos = yes.groupby(["policy_type", "gender"])[
    ["monthly_premium_auto", "customer_lifetime_value", "total_claim_amount"]
].mean().round(2)

# Ratio: valor del cliente por cada euro reclamado (más alto = más rentable / menos riesgo)
segmentos["clv_por_claim"] = (segmentos["customer_lifetime_value"] /
                              segmentos["total_claim_amount"]).round(2)

segmentos = segmentos.sort_values("clv_por_claim", ascending=False)
display(segmentos)

mejor = segmentos.index[0]
peor = segmentos.index[-1]
print(f"Segmento más rentable / menos riesgo: {mejor}")
print(f"Segmento menos rentable / más riesgo: {peor}")

**Conclusión:** la tabla está ordenada por `clv_por_claim` (valor del cliente / reclamaciones). Los segmentos de arriba aportan mucho valor con pocas reclamaciones, así que son los más rentables y de menor riesgo, y conviene dirigir hacia ellos las campañas. Los de abajo reclaman mucho en relación con lo que aportan. Una prima mensual alta suele ir unida a reclamaciones altas, por lo que la prima por sí sola no indica rentabilidad.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [ ]:
# Número de clientes por estado
clientes_estado = df["state"].value_counts()
print(clientes_estado)

# Solo estados con más de 500 clientes
estados_500 = clientes_estado[clientes_estado > 500]
estados_500

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [ ]:
# Máximo, mínimo y mediana del CLV por nivel educativo y género
clv_stats = df.groupby(["education", "gender"])["customer_lifetime_value"].agg(
    ["max", "min", "median"]).round(2)
clv_stats

**Conclusiones:**
- Las medianas son bastante parecidas entre niveles educativos y géneros: la educación y el género no cambian mucho el valor típico de un cliente.
- Los máximos son muchísimo más altos que las medianas: en todos los grupos hay unos pocos clientes de valor muy alto (distribución sesgada a la derecha).
- Los mínimos son similares en todos los grupos (≈1.900–2.000), así que hay un "suelo" de valor común a cualquier cliente.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [ ]:
# Tabla: estados en filas, meses en columnas, número de pólizas en las celdas
policies_state_month = df.pivot_table(index="state", columns="month",
                                      values="policy", aggfunc="count")
policies_state_month

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [ ]:
# 1. Agrupar por estado y mes y contar pólizas
por_estado_mes = df.groupby(["state", "month"])["policy"].count().reset_index(name="policies")

# 2. Los 3 estados con más pólizas en total
top3 = df["state"].value_counts().head(3).index
print("Top 3 estados:", list(top3))

# 3. Filtrar solo esos estados y ponerlo en tabla (meses en columnas)
top3_df = por_estado_mes[por_estado_mes["state"].isin(top3)]
top3_table = top3_df.pivot(index="state", columns="month", values="policies")
top3_table

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

In [ ]:
# Contar respuestas Yes / No por canal de venta (tabla "ancha")
canal_wide = pd.crosstab(df["sales_channel"], df["response"]).reset_index()

# melt: pasar a formato "largo" (una fila por canal y respuesta)
canal_long = canal_wide.melt(id_vars="sales_channel", var_name="response", value_name="customers")
display(canal_long)

# Tasa de respuesta (% de "Yes") por canal
total = canal_long.groupby("sales_channel")["customers"].sum()
yes_count = canal_long[canal_long["response"] == "Yes"].set_index("sales_channel")["customers"]
response_rate = (yes_count / total * 100).round(2).sort_values(ascending=False)
print("Tasa de respuesta (%) por canal:")
response_rate

**Conclusión:** el canal con mayor tasa de respuesta es el primero de la lista. Si la diferencia con los demás es notable, conviene invertir más en ese canal y revisar el de menor tasa.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9